In [1]:
import os
import time
import multiprocessing as mp
import numpy as np
import pandas as pd
from tqdm.auto import tqdm


In [2]:
# --- CONFIGURATION ---
TRAIN_DATAFRAME_PATH_STRING = "data/dataframes/train.csv"


# 14700K = 8 P-cores (16 threads) + 12 E-cores = 28 logical threads.
# For CPU-bound work like this, going much past the physical core count (20)
# tends to add contention rather than throughput. Start here and profile.
NUM_WORKERS = 16

VIDEO_EXTENSIONS = (".mp4", ".avi", ".mov", ".mkv")


BAR_FORMAT = (
    "{desc}: {percentage:3.0f}%|{bar}| {n_fmt}/{total_fmt} "
    "[elapsed: {elapsed}, eta: {remaining}, rate: {rate_fmt}]"
)


In [3]:
df = pd.read_csv(TRAIN_DATAFRAME_PATH_STRING)
df

,file_path,label,id
0,C:\Users\user2\.cache\kagglehub\datasets\mrgei...,after,4a.6001-after-2022_12_14_15_11_39.630-0
1,C:\Users\user2\.cache\kagglehub\datasets\mrgei...,after,4a.6001-after-2022_12_15_13_43_34.681-0
2,C:\Users\user2\.cache\kagglehub\datasets\mrgei...,after,4a.6001-after-2022_12_15_13_53_58.781-0
3,C:\Users\user2\.cache\kagglehub\datasets\mrgei...,after,4a.6001-after-2022_12_16_00_15_45.719-0
4,C:\Users\user2\.cache\kagglehub\datasets\mrgei...,after,4a.6001-after-2022_12_16_00_27_57.274-0
...,...,...,...
30862,C:\Users\user2\.cache\kagglehub\datasets\mrgei...,eye,gtsignstudy4a.8048-eye-2023_01_29_15_35_08.698-0
30863,C:\Users\user2\.cache\kagglehub\datasets\mrgei...,eye,gtsignstudy4a.8048-eye-2023_01_29_16_30_54.043-0
30864,C:\Users\user2\.cache\kagglehub\datasets\mrgei...,eye,gtsignstudy4a.8048-eye-2023_01_29_17_46_14.181-0
30865,C:\Users\user2\.cache\kagglehub\datasets\mrgei...,eye,gtsignstudy4a.8048-eye-2023_01_29_18_03_00.364-0


In [5]:
from modules.landmark_worker import init_worker, process_video

all_video_paths = df["file_path"].to_list()
samples = all_video_paths[:1]

try:
    total_logical = os.cpu_count()
    if total_logical is None:
        raise ValueError("os.cpu_count() returned None")
except Exception as e:
    print(f"Error determining CPU count: {e}")
    total_logical = 1

cpu_percentage = NUM_WORKERS / total_logical * 100


print(
    f"Workers: {NUM_WORKERS} / {total_logical} logical threads available "
    f"({NUM_WORKERS / total_logical:.0%} utilization if fully scheduled)."
)

start_time = time.time()
completed = 0

with mp.Pool(processes=NUM_WORKERS, initializer=init_worker) as pool:
    with tqdm(
        total=len(samples),
        desc="Extracting landmarks",
        unit="video",
        bar_format=BAR_FORMAT,
    ) as pbar:
        for result in pool.imap_unordered(process_video, samples):
            completed += 1
            pbar.set_postfix_str(result, refresh=False)
            pbar.update(1)

elapsed = time.time() - start_time
rate = completed / elapsed if elapsed > 0 else 0.0
print(
    f"Done: {completed}/{len(samples)} videos in {elapsed / 60:.1f} min "
    f"({rate:.2f} videos/sec, {rate * 3600:.0f} videos/hr)."
)


Workers: 16 / 28 logical threads available (57% utilization if fully scheduled).


Extracting landmarks:   0%|          | 0/1 [elapsed: 00:00, eta: ?, rate: ?video/s]

Done: 1/1 videos in 0.3 min (0.07 videos/sec, 236 videos/hr).
